Deliverable 1: G6_2, Tommy Rustad Apelseth, Aron Lerdahl


Task 4: A general function for writing heatwaves to a yml file

In [ ]:
import json
import os
from datetime import date, timedelta

import requests
import yaml

# Reading the client ID from a local file
client_path = os.path.expanduser("~/Documents/client.txt")
with open(client_path, encoding="utf-8") as client_file:
    client_id = client_file.read().strip()


# Function to fetch heatwave data from the API
def get_heatwave_data(source_id, start_date, end_date, client_id):
    endpoint = "https://frost.met.no/observations/v0.jsonld"
    parameters = {
        "sources": source_id,
        "elements": "max(air_temperature P1D)",
        "referencetime": f"{start_date}/{end_date}",
    }

    date_temp = {}
    loop_start_date = date.fromisoformat(start_date)
    loop_end_date = date.fromisoformat(end_date)
    
    r = requests.get(endpoint, params=parameters, auth=(client_id, ""))
    json_data = r.json()

    if r.status_code != 200:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json_data['error']['message'])
        print('Reason: %s' % json_data['error']['reason'])
        return []

    for item in json_data["data"]:
        day = date.fromisoformat(item["referenceTime"][:10])

        for observation in item["observations"]:
            if (
                observation["elementId"] == "max(air_temperature P1D)"
                and observation["timeOffset"] == "PT0H"
                and observation["timeSeriesId"] == 0
            ):
                date_temp[day] = observation["value"]


    print("Number of dates with temperatures:", len(date_temp))
    print("First temperatures:", list(date_temp.items())[:5])

    heatwaves = []
    hot_days = 0
    heatwave_start = None
    threshold = 27  # Temperature threshold for heatwave

    for day in range((loop_end_date - loop_start_date).days + 1):
        current_date = loop_start_date + timedelta(days=day)
        temperature = date_temp.get(current_date)
        
        if temperature is not None and temperature > threshold:
            if hot_days == 0:
                heatwave_start = current_date
            hot_days += 1
        else:
            if hot_days >= 5:
                heatwaves.append((heatwave_start.isoformat(), (current_date - timedelta(days=1)).isoformat(),))
            hot_days = 0

    with open("heatwaves.yml", "w") as file:
        yaml.safe_dump(heatwaves, file, sort_keys=False)

print(get_heatwave_data("SN17850", "2025-01-01", "2025-12-31", client_id))



Number of dates with temperatures: 363
First temperatures: [(datetime.date(2025, 1, 1), -3.4), (datetime.date(2025, 1, 2), -5.2), (datetime.date(2025, 1, 3), -1.4), (datetime.date(2025, 1, 4), -11), (datetime.date(2025, 1, 5), -7.2)]
[('2025-07-16', '2025-07-20')]


In [25]:
import json

endpoint = "https://frost.met.no/observations/v0.jsonld"
parameters = {
    "sources": "SN17850",
    "elements": "max(air_temperature P1D)",
    "referencetime": "2025-01-01/2025-12-31",
}
heatwaves = []
date_temp = {}
loop_start_date = date.fromisoformat("2025-01-01")
loop_end_date = date.fromisoformat("2025-12-31")

r = requests.get(endpoint, params=parameters, auth=(client_id, ""))
json_data = r.json()

with open("data.json", "w") as file:
    json.dump(json_data, file)